In [1]:
from communication.typed_protocol_client import TypedRabbitMQClient
from communication.typed_protocol import LoadProgram, LoadTCPProgram, InjectWear, Play, InjectStuckJoint
from communication.rabbitmq import Rabbitmq
from pathlib import Path
import yaml
import numpy as np

def load_config(path: Path) -> dict:
    with path.open() as f:
        return yaml.safe_load(f)

connect_config = load_config(Path("../../communication/connect.yml"))
connect_config["ip"] = "127.0.0.1"

typed_client = TypedRabbitMQClient(Rabbitmq(**connect_config))
typed_client.client.connect_to_server()

Connected to RabbitMQ server.


In [25]:
# Construct control message for loading a program
def mov_to_pos(position: list, vel: int = 60, acc: int = 80):
    msg = LoadProgram(joint_positions=position, max_velocity=vel, acceleration=acc)

    typed_client.publish(msg)
    # send control message for starting program
    typed_client.publish(Play())

position = [0, -np.pi/2, 0, -np.pi/2, 0, 0]

mov_to_pos(position)

In [8]:
def mov_to_tcp(position: list, rotation: list=[0, 0, np.pi], vel: int = 60, acc: int = 80):
    msg = LoadTCPProgram(tcp_position=position, tcp_rotation=rotation, max_velocity=vel, acceleration=acc)
    typed_client.publish(msg)

mov_to_tcp([0.1, 0.1, -0.3])

In [2]:
def inject_wear():
    msg = InjectWear(duration=100, fault_value=100, joints=[3, 4, 5])
    typed_client.publish(msg)

inject_wear()

In [4]:
def inject_stuck_joints():
    msg = InjectStuckJoint(joints=[0, 1])
    typed_client.publish(msg)

inject_stuck_joints()